# Logistic Regression
Checking the weights for Dry- and Wet-proofing.
What's the difference?

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
data_NL = pd.read_csv('../../../data/raw/SCALAR_Coastal_Study_new_respondents_Wave_Five_NL.csv')

In [4]:
print(data_NL.columns.tolist())

['ID', 'Q1_home_NL_UK', 'Q4_home_size_NL', 'Q5_home_tenure', 'Q5b_home_sell', 'Q6_home_costs', 'Q7_move_in', 'Q8_move_out', 'Q12_neighborhood_trust', 'Q13_neighborhood_community', 'Q14_neighborhood_pleasure', 'Q15_neighborhood_favorite', 'Q16_neighborhood_identity', 'Q11_search_improve', 'Q11_search_social', 'Q11_search_family', 'Q11_search_area', 'Q11_search_job', 'Q11_search_location', 'Q11_search_hazard', 'Q11_search_other', 'Q11_search_dont_know', 'Q11a_hazard_type1', 'Q11a_hazard_type3', 'Q11a_hazard_type4', 'Q11a_hazard_type5', 'Q11a_hazard_type10', 'Q11a_hazard_type6', 'Q11a_hazard_type9', 'Q11a_hazard_not_say', 'R02_perc_prob', 'R02_perc_prob_other_text', 'Q18_flood_exp', 'Q18a_flood_where', 'Q18b_flood_year', 'Q18d_flood_cost', 'R05_worry', 'R01_resilience_5', 'R01_resilience_6', 'Q17_compens_noone', 'Q17_compens_ins', 'Q17_compens_owner', 'Q17_compens_family', 'Q17_compens_ngo', 'Q17_compens_other', 'Q17_compens_dont_know', 'R1a_self_efficacy_SM1', 'R1a_self_efficacy_SM2', 'R

For logistic regression on whether households will take measures, we need the following data:
- threat appraisal:
    * perceived probability (R02_perc_prob)
    * perceived damage / severity (R03_perc_damage) - we don't have this so we estimate it based on the perc probability, worry and flood experience
    * worry (R05_worry)
- coping appraisal
    * perceived cost (R1c_perc_costs)
    * perceived response efficacy (R1b_resp_efficacy)
    * self-efficacy (R1a_self_efficacy)
    * each for the following measures
       * dry-proofing:
          * installing anti-backflow valves on pipes (SM5)
          * installing a pump or similar to drain water (SM6)
          * fixing water barriers (SM7)
       * wet-proofing:
          * strengthening house foundations (SM2)
          * reinforcing walls (SM3)
          * raising electricity meter (SM4)
- other:
    * flood experience
 
And for the y: 
- R2_implementation for relevant measures. Reminder: 1: already implemented, 2: plan to implement in near future (next 1-3 years), rest: implement later or never)


In [5]:
relevant_columns = ['R02_perc_prob', 'R05_worry', # threat appraisal
                    'R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4', 'R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7',                    
                    'R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4', 'R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7', 
                    'R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4', 'R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7', 
                    'Q18_flood_exp',
                    'R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4', 'R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7']

In [6]:
data_households = data_NL[relevant_columns]
data_households.head()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7
0,6,2,1,1,3,3,4,1,2,3,2,3,4,2,4,5,5,4,4,3,0,4,4,4,4,3,4
1,4,1,3,5,5,5,3,3,3,3,3,3,3,3,3,3,3,4,3,4,0,4,4,4,4,4,4
2,2,1,1,1,5,5,1,1,1,1,1,2,3,2,5,5,4,3,4,3,0,4,4,4,4,4,4
3,1,1,1,1,1,1,1,1,3,3,3,3,3,3,5,5,5,5,5,5,0,4,4,4,4,4,4
4,2,1,1,1,2,2,2,1,1,1,1,1,2,1,5,5,3,3,4,3,0,4,4,4,4,4,4


Need to combine some columns. 
- combining dry- and wet-proofing --> averages

In [7]:
data_households['self_efficacy_DP'] = data_households[['R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7']].mean(axis=1)
data_households['self_efficacy_WP'] = data_households[['R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4']].mean(axis=1)
data_households['resp_efficacy_DP'] = data_households[['R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7']].mean(axis=1)
data_households['resp_efficacy_WP'] = data_households[['R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4']].mean(axis=1)
data_households['perc_cost_DP'] = data_households[['R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7']].mean(axis=1)
data_households['perc_cost_WP'] = data_households[['R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4']].mean(axis=1)

/var/folders/qn/rcpbyl8d6rs2ym4z26w2fk7wyq1ksg/T/ipykernel_92333/1443061650.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_households['self_efficacy_DP'] = data_households[['R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7']].mean(axis=1)
/var/folders/qn/rcpbyl8d6rs2ym4z26w2fk7wyq1ksg/T/ipykernel_92333/1443061650.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_households['self_efficacy_WP'] = data_households[['R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_se

In [8]:
data_households.head()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP
0,6,2,1,1,3,3,4,1,2,3,2,3,4,2,4,5,5,4,4,3,0,4,4,4,4,3,4,2.666667,1.666667,3.000000,2.333333,3.666667,4.666667
1,4,1,3,5,5,5,3,3,3,3,3,3,3,3,3,3,3,4,3,4,0,4,4,4,4,4,4,3.666667,4.333333,3.000000,3.000000,3.666667,3.000000
2,2,1,1,1,5,5,1,1,1,1,1,2,3,2,5,5,4,3,4,3,0,4,4,4,4,4,4,2.333333,2.333333,2.333333,1.000000,3.333333,4.666667
3,1,1,1,1,1,1,1,1,3,3,3,3,3,3,5,5,5,5,5,5,0,4,4,4,4,4,4,1.000000,1.000000,3.000000,3.000000,5.000000,5.000000
4,2,1,1,1,2,2,2,1,1,1,1,1,2,1,5,5,3,3,4,3,0,4,4,4,4,4,4,1.666667,1.333333,1.333333,1.000000,3.333333,4.333333


How to combine the implementation for dry- and wet-proofing. Maybe they plan to take some of the dry-proofing emasures but not all. I don't think it makes sense to take the average.
Idea: always take the lowest number available in the three columns that go into the variable (assumption: if they would take 1, they would also do the other)

In [9]:
data_households['implement_DP'] = data_households[['R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7']].min(axis=1)
data_households['implement_WP'] = data_households[['R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4']].min(axis=1)


/var/folders/qn/rcpbyl8d6rs2ym4z26w2fk7wyq1ksg/T/ipykernel_92333/3199941349.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_households['implement_DP'] = data_households[['R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7']].min(axis=1)
/var/folders/qn/rcpbyl8d6rs2ym4z26w2fk7wyq1ksg/T/ipykernel_92333/3199941349.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_households['implement_WP'] = data_households[['R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementati

Cleaning up: removing rows with don't know answers and removing the columns that were used to combine for wet- and dry-proofing just above. 

In [10]:
# remove rows with don't know
data_households_98_removed = data_households[(data_households.R02_perc_prob != 95) &
                                            (data_households.R02_perc_prob != 98) & 
                                            (data_households.R02_perc_prob != 97) & 
                                            #(data_households.R03_perc_damage != 98) &
                                            (data_households.R05_worry != 98)
                                            ]
data_households_98_removed.describe()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
count,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000,359.000000
mean,2.487465,1.568245,1.977716,1.949861,2.247911,2.217270,2.164345,1.877437,2.487465,2.506964,2.590529,2.328691,2.615599,2.493036,4.142061,4.153203,3.554318,3.498607,3.710306,3.632312,0.064067,3.782730,3.752089,3.682451,3.646240,3.707521,3.696379,2.086351,2.058496,2.479109,2.528319,3.613742,3.949861,3.532033,3.601671
std,1.815278,0.871777,1.257182,1.265239,1.444442,1.289247,1.300305,1.233389,1.128223,1.130744,1.226805,1.031913,1.132250,1.157599,1.035346,1.031046,1.170745,1.098182,1.043565,1.102898,0.245214,0.591024,0.653602,0.735758,0.780475,0.713575,0.739744,1.090690,1.158986,0.950079,1.016493,0.970699,0.912680,0.926611,0.842381
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.500000,1.000000,1.000000,2.000000,1.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,1.000000,1.000000,2.000000,2.000000,3.000000,3.333333,4.000000,4.000000
50%,2.000000,1.000000,1.000000,1.000000,2.000000,2.000000,2.000000,1.000000,3.000000,3.000000,3.000000,2.000000,3.000000,3.000000,5.000000,5.000000,4.000000,3.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,1.666667,2.666667,2.666667,3.666667,4.000000,4.000000,4.000000
75%,3.000000,2.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,5.000000,5.000000,5.000000,4.000000,5.000000,5.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,3.000000,3.000000,3.000000,3.000000,4.333333,4.666667,4.000000,4.000000
max,9.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,1.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,4.000000,4.000000


In [11]:
data_households_98_removed = data_households_98_removed.drop(columns = [#'R03_perc_damage_UK1', 'R03_perc_damage_UK2', 'R03_perc_damage_UK3',# threat appraisal
                    'R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4', 'R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7',                    
                    'R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4', 'R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7', 
                    'R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4', 'R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7',
                                          'R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4', 'R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7'
                                                                       ])
data_households_98_removed.head()

,R02_perc_prob,R05_worry,Q18_flood_exp,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
0,6,2,0,2.666667,1.666667,3.000000,2.333333,3.666667,4.666667,3,4
1,4,1,0,3.666667,4.333333,3.000000,3.000000,3.666667,3.000000,4,4
2,2,1,0,2.333333,2.333333,2.333333,1.000000,3.333333,4.666667,4,4
3,1,1,0,1.000000,1.000000,3.000000,3.000000,5.000000,5.000000,4,4
4,2,1,0,1.666667,1.333333,1.333333,1.000000,3.333333,4.333333,4,4


In [12]:
# Initialize MinMaxScaler
scaler = MinMaxScaler()
# Normalize 
data_households_98_removed[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', #'R03_perc_damage', 
                            'self_efficacy_DP', 'self_efficacy_WP', 
                            'resp_efficacy_DP', 'resp_efficacy_WP', 
                            'perc_cost_DP', 'perc_cost_WP']] = scaler.fit_transform(data_households_98_removed[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', #'R03_perc_damage', 
                                                                                                                'self_efficacy_DP', 'self_efficacy_WP', 
                                                                                                                'resp_efficacy_DP', 'resp_efficacy_WP', 
                                                                                                                'perc_cost_DP', 'perc_cost_WP']])
data_households_98_removed.head()

,R02_perc_prob,R05_worry,Q18_flood_exp,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
0,0.625,0.25,0.0,0.416667,0.166667,0.500000,0.333333,0.666667,0.916667,3,4
1,0.375,0.00,0.0,0.666667,0.833333,0.500000,0.500000,0.666667,0.500000,4,4
2,0.125,0.00,0.0,0.333333,0.333333,0.333333,0.000000,0.583333,0.916667,4,4
3,0.000,0.00,0.0,0.000000,0.000000,0.500000,0.500000,1.000000,1.000000,4,4
4,0.125,0.00,0.0,0.166667,0.083333,0.083333,0.000000,0.583333,0.833333,4,4


In [13]:
# binary columns for implementation
data_households_98_removed['done_DP'] = (data_households_98_removed['implement_DP'] == 1).astype(int)
data_households_98_removed['done_WP'] = (data_households_98_removed['implement_WP'] == 1).astype(int)
data_households_98_removed['plan_soon_DP'] = (data_households_98_removed['implement_DP'] == 2).astype(int)
data_households_98_removed['plan_soon_WP'] = (data_households_98_removed['implement_WP'] == 2).astype(int)
data_households_98_removed.head(10)

,R02_perc_prob,R05_worry,Q18_flood_exp,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP,done_DP,done_WP,plan_soon_DP,plan_soon_WP
0,0.625,0.25,0.0,0.416667,0.166667,0.500000,0.333333,0.666667,0.916667,3,4,0,0,0,0
1,0.375,0.00,0.0,0.666667,0.833333,0.500000,0.500000,0.666667,0.500000,4,4,0,0,0,0
2,0.125,0.00,0.0,0.333333,0.333333,0.333333,0.000000,0.583333,0.916667,4,4,0,0,0,0
3,0.000,0.00,0.0,0.000000,0.000000,0.500000,0.500000,1.000000,1.000000,4,4,0,0,0,0
4,0.125,0.00,0.0,0.166667,0.083333,0.083333,0.000000,0.583333,0.833333,4,4,0,0,0,0
6,0.000,0.25,0.0,0.416667,0.416667,0.500000,0.500000,1.000000,1.000000,3,3,0,0,0,0
7,0.000,0.00,0.0,0.000000,0.000000,0.166667,0.000000,0.166667,0.000000,4,4,0,0,0,0
8,0.000,0.00,0.0,0.500000,0.000000,0.333333,0.416667,0.500000,0.750000,4,4,0,0,0,0
9,0.125,0.00,0.0,0.500000,0.500000,0.000000,0.166667,0.750000,0.750000,4,4,0,0,0,0
11,0.125,0.00,0.0,0.500000,0.250000,0.333333,0.333333,0.916667,0.916667,4,4,0,0,0,0


In [14]:
# add column on perceived damage (see estimate_severity_NL.ipynb for equation generation)
data_households_98_removed['perc_damage'] = 0.4302 + (0.2665 * data_households_98_removed['R02_perc_prob']) + (0.1581 * data_households_98_removed['R05_worry']) + (-0.0871 * data_households_98_removed['Q18_flood_exp'])
data_households_98_removed.head()

,R02_perc_prob,R05_worry,Q18_flood_exp,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP,done_DP,done_WP,plan_soon_DP,plan_soon_WP,perc_damage
0,0.625,0.25,0.0,0.416667,0.166667,0.500000,0.333333,0.666667,0.916667,3,4,0,0,0,0,0.636288
1,0.375,0.00,0.0,0.666667,0.833333,0.500000,0.500000,0.666667,0.500000,4,4,0,0,0,0,0.530138
2,0.125,0.00,0.0,0.333333,0.333333,0.333333,0.000000,0.583333,0.916667,4,4,0,0,0,0,0.463513
3,0.000,0.00,0.0,0.000000,0.000000,0.500000,0.500000,1.000000,1.000000,4,4,0,0,0,0,0.430200
4,0.125,0.00,0.0,0.166667,0.083333,0.083333,0.000000,0.583333,0.833333,4,4,0,0,0,0,0.463513


In [15]:
print(data_households_98_removed['done_DP'].value_counts())
print(data_households_98_removed['done_WP'].value_counts())
print(data_households_98_removed['plan_soon_DP'].value_counts())
print(data_households_98_removed['plan_soon_WP'].value_counts())

done_DP
0    335
1     24
Name: count, dtype: int64
done_WP
0    342
1     17
Name: count, dtype: int64
plan_soon_DP
0    322
1     37
Name: count, dtype: int64
plan_soon_WP
0    326
1     33
Name: count, dtype: int64


In [16]:
data_DP_WP_reg = data_households_98_removed

In [17]:
data_DP_WP_reg.count()

R02_perc_prob       359
R05_worry           359
Q18_flood_exp       359
self_efficacy_DP    359
self_efficacy_WP    359
resp_efficacy_DP    359
resp_efficacy_WP    359
perc_cost_DP        359
perc_cost_WP        359
implement_DP        359
implement_WP        359
done_DP             359
done_WP             359
plan_soon_DP        359
plan_soon_WP        359
perc_damage         359
dtype: int64

### Do the logistic regression

We do the logistic regression 4 times here. 
1. logistic regression to done_WP, so to the ones that have already taken the measure WP
2. logistic regression to plan_soon_WP: those who plan to take the measure in the next 6 months
3. logistic regression to done_DP, so to the ones that have already taken the measure DP
4. logistic regression to plan_soon_DP: those who plan to take the measure in the next 6 months

In [18]:
# Wet-proofing
# logistic regression to 'done_WP'
# Define features (X) and target (y)
X_WP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP']]
y_WP_done = data_DP_WP_reg['done_WP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_WP_done, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_WP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.97

Logistic Regression Equation:
logit(p) = -2.5951 (-0.0555 * R02_perc_prob) + (1.3550 * R05_worry) + (0.3116 * Q18_flood_exp) + (0.1633 * perc_damage) + (0.8020 * self_efficacy_WP) + (-0.6450 * resp_efficacy_WP) + (-1.4335 * perc_cost_WP) + (1.7353 * done_DP)

Feature Weights: {'R02_perc_prob': -0.055485340455925194, 'R05_worry': 1.3549613163514675, 'Q18_flood_exp': 0.31164138385477286, 'perc_damage': 0.1632696592929165, 'self_efficacy_WP': 0.8019974617892163, 'resp_efficacy_WP': -0.6449686260327634, 'perc_cost_WP': -1.4334543097143253, 'done_DP': 1.7353081176290384}
Intercept: -2.595105983792344


In [19]:
LR_values_done_WP = dict(zip(feature_names, weights))
LR_values_done_WP['Intercept'] = intercept
LR_values_done_WP['Perc_probability'] = LR_values_done_WP['R02_perc_prob']
LR_values_done_WP['worry'] = LR_values_done_WP['R05_worry']
LR_values_done_WP['flood_experience'] = LR_values_done_WP['Q18_flood_exp']
LR_values_done_WP['perc_damage'] = LR_values_done_WP['perc_damage']
LR_values_done_WP['self_efficacy'] = LR_values_done_WP['self_efficacy_WP']
LR_values_done_WP['resp_efficacy'] = LR_values_done_WP['resp_efficacy_WP']
LR_values_done_WP['perc_cost'] = LR_values_done_WP['perc_cost_WP']
LR_values_done_WP['done_other'] = LR_values_done_WP['done_DP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP')
for k in remove:
    LR_values_done_WP.pop(k, None)
LR_values_done_WP

{'perc_damage': 0.1632696592929165,
 'Intercept': -2.595105983792344,
 'Perc_probability': -0.055485340455925194,
 'worry': 1.3549613163514675,
 'flood_experience': 0.31164138385477286,
 'self_efficacy': 0.8019974617892163,
 'resp_efficacy': -0.6449686260327634,
 'perc_cost': -1.4334543097143253,
 'done_other': 1.7353081176290384}

In [20]:
# Wet-proofing
# logistic regression to 'plan_soon_WP'
# Define features (X) and target (y)
X_WP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP']]
y_WP_plan = data_DP_WP_reg['plan_soon_WP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_WP_plan, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_WP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.86

Logistic Regression Equation:
logit(p) = -3.0659 (0.9357 * R02_perc_prob) + (0.3283 * R05_worry) + (1.0009 * Q18_flood_exp) + (0.2062 * perc_damage) + (1.7956 * self_efficacy_WP) + (0.6176 * resp_efficacy_WP) + (-1.2643 * perc_cost_WP) + (0.5842 * done_DP)

Feature Weights: {'R02_perc_prob': 0.9357484557535579, 'R05_worry': 0.328326321859738, 'Q18_flood_exp': 1.0009069693592572, 'perc_damage': 0.2061506611745964, 'self_efficacy_WP': 1.7956245633276973, 'resp_efficacy_WP': 0.6175772419438644, 'perc_cost_WP': -1.2642747592660146, 'done_DP': 0.5841862592825637}
Intercept: -3.065913254294365


In [21]:
LR_values_plan_soon_WP = dict(zip(feature_names, weights))
LR_values_plan_soon_WP['Intercept'] = intercept
LR_values_plan_soon_WP['Perc_probability'] = LR_values_plan_soon_WP['R02_perc_prob']
LR_values_plan_soon_WP['worry'] = LR_values_plan_soon_WP['R05_worry']
LR_values_plan_soon_WP['flood_experience'] = LR_values_plan_soon_WP['Q18_flood_exp']
LR_values_plan_soon_WP['perc_damage'] = LR_values_plan_soon_WP['perc_damage']
LR_values_plan_soon_WP['self_efficacy'] = LR_values_plan_soon_WP['self_efficacy_WP']
LR_values_plan_soon_WP['resp_efficacy'] = LR_values_plan_soon_WP['resp_efficacy_WP']
LR_values_plan_soon_WP['perc_cost'] = LR_values_plan_soon_WP['perc_cost_WP']
LR_values_plan_soon_WP['done_other'] = LR_values_plan_soon_WP['done_DP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP')
for k in remove:
    LR_values_plan_soon_WP.pop(k, None)
LR_values_plan_soon_WP

{'perc_damage': 0.2061506611745964,
 'Intercept': -3.065913254294365,
 'Perc_probability': 0.9357484557535579,
 'worry': 0.328326321859738,
 'flood_experience': 1.0009069693592572,
 'self_efficacy': 1.7956245633276973,
 'resp_efficacy': 0.6175772419438644,
 'perc_cost': -1.2642747592660146,
 'done_other': 0.5841862592825637}

In [22]:
# Dry-proofing
# logistic regression to 'done_DP'
# Define features (X) and target (y)
X_DP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP']]
y_DP_done = data_DP_WP_reg['done_DP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_DP_done, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_DP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 1.00

Logistic Regression Equation:
logit(p) = -3.8994 (-0.1313 * R02_perc_prob) + (0.4907 * R05_worry) + (0.5899 * Q18_flood_exp) + (-0.0122 * perc_damage) + (0.5486 * self_efficacy_DP) + (0.1229 * resp_efficacy_DP) + (-0.5744 * perc_cost_DP) + (4.7455 * done_WP)

Feature Weights: {'R02_perc_prob': -0.13134156085765358, 'R05_worry': 0.49071496499739886, 'Q18_flood_exp': 0.5899055371222234, 'perc_damage': -0.012204735192746148, 'self_efficacy_DP': 0.548640347142407, 'resp_efficacy_DP': 0.12290138607921942, 'perc_cost_DP': -0.5744147409826644, 'done_WP': 4.745511697261656}
Intercept: -3.8993947312783224


In [23]:
LR_values_done_DP = dict(zip(feature_names, weights))
LR_values_done_DP['Intercept'] = intercept
LR_values_done_DP['Perc_probability'] = LR_values_done_DP['R02_perc_prob']
LR_values_done_DP['worry'] = LR_values_done_DP['R05_worry']
LR_values_done_DP['flood_experience'] = LR_values_done_DP['Q18_flood_exp']
LR_values_done_DP['perc_damage'] = LR_values_done_DP['perc_damage']
LR_values_done_DP['self_efficacy'] = LR_values_done_DP['self_efficacy_DP']
LR_values_done_DP['resp_efficacy'] = LR_values_done_DP['resp_efficacy_DP']
LR_values_done_DP['perc_cost'] = LR_values_done_DP['perc_cost_DP']
LR_values_done_DP['done_other'] = LR_values_done_DP['done_WP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP')
for k in remove:
    LR_values_done_DP.pop(k, None)
LR_values_done_DP

{'perc_damage': -0.012204735192746148,
 'Intercept': -3.8993947312783224,
 'Perc_probability': -0.13134156085765358,
 'worry': 0.49071496499739886,
 'flood_experience': 0.5899055371222234,
 'self_efficacy': 0.548640347142407,
 'resp_efficacy': 0.12290138607921942,
 'perc_cost': -0.5744147409826644,
 'done_other': 4.745511697261656}

In [24]:
# Dry-proofing
# logistic regression to 'plan_soon_DP'
# Define features (X) and target (y)
X_DP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP']]
y_DP_plan = data_DP_WP_reg['plan_soon_DP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_DP_plan, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_DP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.86

Logistic Regression Equation:
logit(p) = -2.1482 (0.8191 * R02_perc_prob) + (0.4690 * R05_worry) + (1.1713 * Q18_flood_exp) + (0.1896 * perc_damage) + (1.5589 * self_efficacy_DP) + (-0.0070 * resp_efficacy_DP) + (-1.5259 * perc_cost_DP) + (-1.3305 * done_WP)

Feature Weights: {'R02_perc_prob': 0.8190719057590434, 'R05_worry': 0.4690096852910323, 'Q18_flood_exp': 1.1713188881440473, 'perc_damage': 0.1895955025852892, 'self_efficacy_DP': 1.5589143205150549, 'resp_efficacy_DP': -0.00697477069670919, 'perc_cost_DP': -1.5258781564618427, 'done_WP': -1.3304503769276812}
Intercept: -2.148202844906897


In [25]:
LR_values_plan_soon_DP = dict(zip(feature_names, weights))
LR_values_plan_soon_DP['Intercept'] = intercept
LR_values_plan_soon_DP['Perc_probability'] = LR_values_plan_soon_DP['R02_perc_prob']
LR_values_plan_soon_DP['worry'] = LR_values_plan_soon_DP['R05_worry']
LR_values_plan_soon_DP['flood_experience'] = LR_values_plan_soon_DP['Q18_flood_exp']
LR_values_plan_soon_DP['perc_damage'] = LR_values_plan_soon_DP['perc_damage']
LR_values_plan_soon_DP['self_efficacy'] = LR_values_plan_soon_DP['self_efficacy_DP']
LR_values_plan_soon_DP['resp_efficacy'] = LR_values_plan_soon_DP['resp_efficacy_DP']
LR_values_plan_soon_DP['perc_cost'] = LR_values_plan_soon_DP['perc_cost_DP']
LR_values_plan_soon_DP['done_other'] = LR_values_plan_soon_DP['done_WP']

remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP')
for k in remove:
    LR_values_plan_soon_DP.pop(k, None)
LR_values_plan_soon_DP

{'perc_damage': 0.1895955025852892,
 'Intercept': -2.148202844906897,
 'Perc_probability': 0.8190719057590434,
 'worry': 0.4690096852910323,
 'flood_experience': 1.1713188881440473,
 'self_efficacy': 1.5589143205150549,
 'resp_efficacy': -0.00697477069670919,
 'perc_cost': -1.5258781564618427,
 'done_other': -1.3304503769276812}

In [26]:
# combining the dictionaries to a dataframe
LogisticReg_PMT = pd.DataFrame([LR_values_done_WP, LR_values_plan_soon_WP, LR_values_done_DP, LR_values_plan_soon_DP], index=['done_WP', 'plan_soon_WP', 'done_DP', 'plan_soon_DP'])

# Transpose the DataFrame to have keys as index and dictionary names as columns
LogisticReg_PMT = LogisticReg_PMT.T

# Define the desired order of the index
desired_order = ['Intercept', 'worry', 'perc_damage', 'Perc_probability', 'flood_experience', 
                 'self_efficacy', 'resp_efficacy', 
                 'perc_cost', 'done_other'
                ]

# Reindex the DataFrame
LogisticReg_PMT = LogisticReg_PMT.reindex(desired_order)

LogisticReg_PMT

,done_WP,plan_soon_WP,done_DP,plan_soon_DP
Intercept,-2.595106,-3.065913,-3.899395,-2.148203
worry,1.354961,0.328326,0.490715,0.469010
perc_damage,0.163270,0.206151,-0.012205,0.189596
Perc_probability,-0.055485,0.935748,-0.131342,0.819072
flood_experience,0.311641,1.000907,0.589906,1.171319
self_efficacy,0.801997,1.795625,0.548640,1.558914
resp_efficacy,-0.644969,0.617577,0.122901,-0.006975
perc_cost,-1.433454,-1.264275,-0.574415,-1.525878
done_other,1.735308,0.584186,4.745512,-1.330450


In [27]:
LogisticReg_PMT_done = LogisticReg_PMT[['done_WP', 'done_DP']]
LogisticReg_PMT_done = LogisticReg_PMT_done.rename(columns={"done_WP": "wet-proofing", "done_DP": "dry-proofing"})
LogisticReg_PMT_plan_soon = LogisticReg_PMT[['plan_soon_WP', 'plan_soon_DP']]
LogisticReg_PMT_plan_soon = LogisticReg_PMT_plan_soon.rename(columns={"plan_soon_WP": "wet-proofing", "plan_soon_DP": "dry-proofing"})
LogisticReg_PMT_done

,wet-proofing,dry-proofing
Intercept,-2.595106,-3.899395
worry,1.354961,0.490715
perc_damage,0.163270,-0.012205
Perc_probability,-0.055485,-0.131342
flood_experience,0.311641,0.589906
self_efficacy,0.801997,0.548640
resp_efficacy,-0.644969,0.122901
perc_cost,-1.433454,-0.574415
done_other,1.735308,4.745512


In [28]:
LogisticReg_PMT_done.to_csv('../../../data/processed/logistic_regression_PMT/Logistic_regression_PMT_done_NLw5.csv')
LogisticReg_PMT_plan_soon.to_csv('../../../data/processed/logistic_regression_PMT/Logistic_regression_PMT_plan_soon_NLw5.csv')